In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    doc["module"] = doc["filename"].split("/")[0]
    documents.append(doc)

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q1. How many lesson pages ?

In [4]:
print(len(files))

72


## Q2. Indexing and searching

In [5]:
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

index = build_index(documents)

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query)
print(search_results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

In [7]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [8]:
from openai import OpenAI
from dotenv import load_dotenv
from rag_base.rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

In [9]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, response = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

The loop keeps calling the model by:

1. Sending the current `messages` history to the model.
2. Checking the response for any `function_call` items.
3. Running each tool call and appending the tool output to `messages`.
4. Repeating the API call in a `while True` loop.

It stops when a model response has no function calls:

```python
if has_function_calls == False:
    break
```

So the exit condition is simply: no tool calls this turn means the agent is done.


In [10]:
print(response.usage.input_tokens)

7121


## Q4. Chunking

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
len(chunks)

295

## Q5. RAG with chunking

In [13]:
index = build_index(chunks)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, response = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

The loop keeps calling the model inside a `while True` loop. Each turn it:

1. Sends the current `messages` to the model.
2. Checks the response for any `function_call` items.
3. Runs those tool calls and appends the results back to `messages`.
4. Sets `has_function_calls = True` if any tool was used.

At the end of the turn, it breaks only if `has_function_calls == False`, meaning the model returned a final answer with no more tool calls.


In [14]:
print(response.usage.input_tokens)

2304


## Q6. Turning it into an agent

In [15]:
import json

from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [16]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lesson contents to answer the user's query.
    """
    return index.search(
        query,
        num_results=5
    )

In [17]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [18]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': "Search the course lesson contents to answer the user's query.",
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [19]:
instructions = """You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
